## 01 · İlanların ortak şablona dönüştürülmesi
Kaynak: Trendyol, Insider One ve Midas (Lever) ile Dream Games (Greenhouse) kariyer sayfalarındaki ilanlar; işe alım sistemlerinin herkese açık ilan uçlarından alındı. İki sistem aynı bilgiyi farklı alanlarda ve farklı yazımlarla veriyor. İlanda açıkça yazmayan bilgi tahmin edilmez, boş bırakılır.

In [1]:
# Ortam hazırlığı: Colab'da repo klonlanır, yerelde notebooks/ klasöründen proje köküne geçilir
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("job-opportunity-mcp-agent").exists() and not Path("README.md").exists():
    !git clone -q https://github.com/alimdemir/job-opportunity-mcp-agent.git
if IN_COLAB and Path("job-opportunity-mcp-agent").exists():
    %cd job-opportunity-mcp-agent
    !pip -q install -r requirements.txt
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("çalışma klasörü:", Path.cwd().name)

çalışma klasörü: job-opportunity-mcp-agent


In [2]:
import pandas as pd
from jobs import load_posts, normalize

posts = load_posts()
print(len(posts), "ilan:", pd.Series([p["company"] for p in posts]).value_counts().to_dict())

# Aynı bilgi iki sistemde farklı yerlerde duruyor
lever = next(p for p in posts if p["source"] == "lever")["raw"]
gh = next(p for p in posts if p["source"] == "greenhouse")["raw"]
print("Lever     :", {k: lever["categories"][k] for k in ("location", "commitment")}, "| workplaceType:", lever["workplaceType"])
print("Greenhouse:", {"location": gh["location"]["name"], "departments": gh["departments"]}, "| çalışma biçimi alanı yok")

14 ilan: {'Trendyol': 5, 'Midas': 4, 'Insider One': 4, 'Dream Games': 1}
Lever     : {'location': 'İstanbul, Turkey', 'commitment': 'Full-Time'} | workplaceType: remote
Greenhouse: {'location': 'İstanbul', 'departments': ['Marketing']} | çalışma biçimi alanı yok


In [3]:
jobs = [normalize(p) for p in posts]
df = pd.DataFrame([j.model_dump() for j in jobs])
df["role"] = df["role"].str.slice(0, 34)
df["tech_stack"] = df["tech_stack"].apply(lambda xs: ", ".join(xs[:3]) + (" ..." if len(xs) > 3 else ""))
df[["company", "role", "location", "country", "work_mode", "employment_type", "tech_stack"]]

,company,role,location,country,work_mode,employment_type,tech_stack
0,Midas,Software Engineer,"İstanbul, Turkey",Türkiye,remote,Tam zamanlı,
1,Midas,Senior Software Engineer,Turkey,Türkiye,remote,Tam zamanlı,
2,Midas,Quantitative Developer,"İstanbul, Turkey",Türkiye,remote,Tam zamanlı,"Python, Go"
3,Midas,Analytics Manager,"İstanbul, Turkey",Türkiye,hybrid,Tam zamanlı,SQL
4,Insider One,Software Engineer - (AI Native),Turkey,Türkiye,remote,Tam zamanlı,"Python, TypeScript, PHP ..."
5,Insider One,Information Security Engineer - (R,Turkey,Türkiye,remote,Tam zamanlı,"AWS, LLM"
6,Insider One,AI Product Manager - B2B SaaS (Mar,Istanbul,Türkiye,remote,Tam zamanlı,"LLM, RAG"
7,Insider One,Technical Support Specialist (Germ,"Istanbul, Turkiye",Türkiye,onsite,Tam zamanlı,
8,Trendyol,Data Scientist (Pre-Search),Istanbul / Maslak,Türkiye,hybrid,Tam zamanlı,"Python, SQL, PyTorch ..."
9,Trendyol,Fraud Process & Analytics Professi,Istanbul / Maslak,Türkiye,hybrid,Tam zamanlı,"SQL, BigQuery"


In [4]:
# İlanda yazmayan alanlar tahmin edilmedi, boş bırakıldı
df[["country", "work_mode", "employment_type", "salary"]].isna().sum().rename("boş alan sayısı")

country             0
work_mode           1
employment_type     1
salary             14
Name: boş alan sayısı, dtype: int64

In [5]:
# İlan metninden çıkarılan şartlar (ön kontrolde kullanılacak)
sart = pd.DataFrame([{"id": j.id, "şirket": j.company, "rol": j.role[:40],
                      "istenen dil": ", ".join(j.required_languages) or "-",
                      "vatandaşlık şartı": j.nationality_only or "-"} for j in jobs])
sart[(sart["istenen dil"] != "-") | (sart["vatandaşlık şartı"] != "-")]

,id,şirket,rol,istenen dil,vatandaşlık şartı
6,7,Insider One,AI Product Manager - B2B SaaS (Martech),"Türkçe, İngilizce",-
7,8,Insider One,Technical Support Specialist (German Spe,"Almanca, İngilizce",-
8,9,Trendyol,Data Scientist (Pre-Search),Arapça,-
11,12,Trendyol,Business Development / Category Manageme,İngilizce,Suudi Arabistan
